# LLM Quantization with AWQ (Activation-aware Weight Quantization)

This notebook quantizes `gpt2-medium` to **4-bit AWQ** format using the `autoawq` library.

**Pipeline:**
1. Install dependencies
2. Load tokenizer + model via AutoAWQ
3. Define calibration texts (used to observe activations)
4. Run AWQ quantization
5. Save quantized model + tokenizer
6. Reload and verify with inference

**Requirements:** GPU with ≥8GB VRAM (T4 on Colab is sufficient)

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# autoawq: AWQ quantization library
# transformers: model/tokenizer loading and inference
# accelerate: device management helpers used internally by autoawq
!pip install -q -U autoawq transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.4 MB/s eta 0:00:00


In [7]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import os
import gc
import torch
from awq import AutoAWQForCausalLM          # AWQ model wrapper
from transformers import AutoTokenizer, AutoConfig

# Suppress tokenizer parallelism warning in notebook environments
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Disable PyTorch's Scaled Dot-Product Attention (SDPA) backend.
# AWQ's calibration hooks patch individual attention ops; SDPA fuses
# them into a single kernel, which breaks the hooks and causes errors.
os.environ["PYTORCH_USE_SDPA"] = "0"

# ── Sanity-check the environment ──────────────────────────────────────────────
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. AWQ quantization requires a CUDA GPU.")

PyTorch version : 2.10.0+cu128
CUDA available  : True
CUDA version    : 12.8
GPU             : Tesla T4


In [9]:
# ── Cell 3: Configuration ──────────────────────────────────────────────────────

# Source model to quantize (change this to any causal LM on HuggingFace)
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
OUT_DIR    = "qwen2.5-0.5b-awq"


# AWQ quantization settings
# zero_point : use asymmetric quantization (recommended, improves quality)
# q_group_size: number of weights per quantization group (128 is standard)
# w_bit       : quantize weights to 4 bits
# version     : GEMM is the fast matrix-multiply kernel (vs GEMV for very small batch)
QUANT_CONFIG = {
    "zero_point"  : True,
    "q_group_size": 128,
    "w_bit"       : 4,
    "version"     : "GEMM",
}

In [10]:
# ── Cell 4: Clear GPU memory before loading ────────────────────────────────────
# Good practice before loading a large model to avoid OOM from stale tensors
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("GPU memory cleared.")

GPU memory cleared.


In [11]:
# ── Cell 5: Load tokenizer ─────────────────────────────────────────────────────
print(f"Loading tokenizer from '{BASE_MODEL}'...")
tok = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True,        # Use the Rust-based fast tokenizer
    trust_remote_code=True,
)

# GPT-2 has no pad token by default; AWQ's calibration loop needs one
# to batch-pad sequences to equal length during calibration.
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Tokenizer loaded.")

Loading tokenizer from 'Qwen/Qwen2.5-0.5B-Instruct'...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded.


In [12]:
# ── Cell 6: Load model via AutoAWQ ────────────────────────────────────────────
# AutoAWQForCausalLM wraps the standard HuggingFace model and adds
# AWQ-specific methods: .quantize() and .save_quantized().
#
# Key arguments:
#   low_cpu_mem_usage : load weights shard-by-shard to avoid CPU OOM
#   use_cache=False   : disable KV-cache (not needed during quantization)
#   torch_dtype       : load in fp16 to fit in GPU VRAM; AWQ will then
#                       quantize the weights down to int4
#   device_map        : pin everything to GPU 0 if CUDA is available
print(f"Loading model '{BASE_MODEL}' via AutoAWQ...")
mdl = AutoAWQForCausalLM.from_pretrained(
    BASE_MODEL,
    low_cpu_mem_usage=True,
    use_cache=False,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map={"": 0} if torch.cuda.is_available() else "cpu",
)

# Set to eval mode: disables dropout and batch-norm update, which would
# distort the activation statistics AWQ relies on for calibration.
mdl.eval()
print("Model loaded and set to eval mode.")

Loading model 'Qwen/Qwen2.5-0.5B-Instruct' via AutoAWQ...


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded and set to eval mode.


In [20]:
import pandas as pd
# ── Cell 7: Calibration data ───────────────────────────────────────────────────
# AWQ needs a small set of representative text samples to observe which
# weight channels are most activated.  It then scales those channels
# before rounding, preserving accuracy despite 4-bit precision.
#
# Rules of thumb:
#   - 50–512 samples is typical; more → better quality, slower calibration
#   - Texts should resemble your target domain
#   - AutoAWQ's .quantize() accepts a plain list[str] — do NOT pre-tokenize
print("Preparing calibration data...")
df = pd.read_csv("/content/medquad.csv")
print(df.columns.tolist())
CALIB_TEXTS = (
    df["answer"]          # ← replace with your actual column name
    .dropna()             # remove any NaN rows that would crash tokenization
    .astype(str)          # ensure every entry is a plain string
    .head(100)            # only need 50 samples for calibration
    .tolist()             # convert to list[str] — the format AWQ expects
)

print(f"{len(CALIB_TEXTS)} calibration samples ready.")
print("Sample:", CALIB_TEXTS[0][:100])  # preview first sample

Preparing calibration data...
['question', 'answer']
100 calibration samples ready.
Sample: Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and 


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [21]:
# ── Cell 8: Run AWQ quantization ───────────────────────────────────────────────
# .quantize() performs three steps internally:
#   1. Forward pass on calib_data to collect per-channel activation scales
#   2. Scale weight channels to minimize quantization error (the AWQ trick)
#   3. Round weights to int4 using the GEMM kernel format
#
# Parameters:
#   tok                    : tokenizer used to encode calib_data internally
#   quant_config           : dict with w_bit, q_group_size, etc.
#   calib_data             : list[str] — raw text, NOT pre-tokenized tensors
#   max_calib_seq_len      : truncate/pad each sample to this many tokens
#   max_calib_samples      : maximum number of samples to process
#   n_parallel_calib_samples: how many samples to process simultaneously;
#                            set to 1 to minimize VRAM usage on small GPUs
print("Starting AWQ quantization...")

mdl.quantize(
    tok,
    quant_config=QUANT_CONFIG,
    calib_data=CALIB_TEXTS,          # ✅ plain list[str] — correct format
    max_calib_seq_len=128,            # keeps memory low; increase for better quality
    max_calib_samples=100,
    n_parallel_calib_samples=1,       # sequential = minimal VRAM
)

print("✅ Quantization complete!")

Starting AWQ quantization...


AWQ: 100%|██████████| 24/24 [02:31<00:00,  6.32s/it]

✅ Quantization complete!


In [22]:
# ── Cell 9: Save quantized model ──────────────────────────────────────────────
# save_quantized() writes the int4 weight tensors in safetensors format
# along with an updated config that records the quantization parameters.
# We also save the tokenizer so the output directory is self-contained.
print(f"Saving quantized model to '{OUT_DIR}'...")

mdl.save_quantized(OUT_DIR, safetensors=True)   # saves model weights + quant config
tok.save_pretrained(OUT_DIR)                    # saves tokenizer files

# Save the base model config so the directory is fully self-contained.
# Note: save_quantized already writes a modified config; this call
# ensures the original architecture fields are present too.
cfg = AutoConfig.from_pretrained(BASE_MODEL, trust_remote_code=True)
cfg.save_pretrained(OUT_DIR)

print(f"✅ Model saved to '{OUT_DIR}'")
print("Contents:")
for f in sorted(os.listdir(OUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
    print(f"  {f:<40} {size_mb:>8.2f} MB")

Saving quantized model to 'qwen2.5-0.5b-awq'...


Writing model shards: 0it [00:00, ?it/s]

✅ Model saved to 'qwen2.5-0.5b-awq'
Contents:
  chat_template.jinja                          0.00 MB
  config.json                                  0.00 MB
  generation_config.json                       0.00 MB
  model.safetensors                          458.38 MB
  tokenizer.json                              11.42 MB
  tokenizer_config.json                        0.00 MB


In [23]:
# ── Cell 10: Free memory before reload ────────────────────────────────────────
# Delete the quantization-time model object and flush GPU memory
# so we start the inference test with a clean slate.
del mdl
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("Memory freed.")

Memory freed.


In [25]:
# ── Cell 11: Reload quantized model and run inference ─────────────────────────
# from_quantized() reads the int4 weights and reconstructs a model
# that can run fast AWQ inference via fused GEMM kernels.
print(f"Reloading quantized model from '{OUT_DIR}'...")

mdl_q = AutoAWQForCausalLM.from_quantized(
    OUT_DIR,
    fuse_layers=True,          # fuse attention + MLP layers for faster inference
    trust_remote_code=True,
    device_map={"": 0} if torch.cuda.is_available() else "cpu",
)
tok_q = AutoTokenizer.from_pretrained(OUT_DIR)
if tok_q.pad_token is None:
    tok_q.pad_token = tok_q.eos_token

# ── Run a simple text generation test ─────────────────────────────────────────
prompt = "What are risks of High Blood pressure"
inputs = tok_q(prompt, return_tensors="pt").to(
    next(mdl_q.parameters()).device
)

with torch.no_grad():
    output_ids = mdl_q.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,         # greedy decoding for reproducibility
        temperature=1.0,
        repetition_penalty=1.1,
    )

# Decode only the newly generated tokens (skip the prompt)
generated = tok_q.decode(
    output_ids[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print("\n" + "=" * 60)
print(f"Prompt   : {prompt}")
print(f"Generated: {generated}")
print("=" * 60)
print("\n✅ AWQ quantization pipeline complete!")

Reloading quantized model from 'qwen2.5-0.5b-awq'...


Replacing layers...: 100%|██████████| 24/24 [00:08<00:00,  2.85it/s]
/usr/local/lib/python3.12/dist-packages/awq/models/base.py:541: UserWarning: Skipping fusing modules because AWQ extension is not installed.No module named 'awq_ext'
  warnings.warn("Skipping fusing modules because AWQ extension is not installed." + msg)
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



Prompt   : What are risks of High Blood pressure
Generated: ?
High blood pressure is a major risk factor for the development and progression of cardiovascular disease. The American Heart Association (AHA) recommends that people with high blood pressure have their blood pressure controlled to keep it in the normal range.
The AHA states that:
- People who have high blood pressure should be monitored regularly by a physician or other health care provider
- People with high blood pressure may need to take medicine to control their blood pressure
- People with high blood pressure should avoid smoking, alcohol use, and excessive weight
- People with high blood pressure should eat a healthy diet
- People with high blood pressure should get regular exercise

✅ AWQ quantization pipeline complete!
